# SoulChat Standalone Kaggle Inference Server

Chỉ load `scutcyr/SoulChat` và expose `/v1/generate` qua ngrok. Notebook này chỉ chạy model baseline độc lập và expose HTTP API.


In [1]:
# 1. Install dependencies. ChatGLM/SoulChat thường ổn hơn với transformers 4.30.x.
!pip install -q transformers==4.30.2 accelerate bitsandbytes==0.37.2 pyngrok fastapi uvicorn nest_asyncio sentencepiece==0.1.99 protobuf==3.20.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 36.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 19.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 104.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 MB 22.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.6 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a pro

In [2]:
# 2. Runtime setup
import os
import gc
import torch
import nest_asyncio
from pyngrok import ngrok

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
nest_asyncio.apply()

if not torch.cuda.is_available():
    print("WARNING: GPU is not enabled. Turn on GPU in Kaggle Accelerator settings.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")


ModuleNotFoundError: No module named 'pyngrok'

In [ ]:
# 3. Load SoulChat model only
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = os.getenv("SOULCHAT_MODEL_NAME", "scutcyr/SoulChat")
TOKENIZER_NAME = os.getenv("SOULCHAT_TOKENIZER_NAME", "THUDM/chatglm-6b")
print(f"Loading tokenizer={TOKENIZER_NAME}, model={MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("SoulChat ready.")


In [ ]:
# 4. Prompt parsing + standalone FastAPI server
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI(title="SoulChat Standalone Inference")

def parse_prompt(full_prompt: str):
    # Expected benchmark prompt format:
    # Lịch sử:
...
User: latest message
SoulChat:
    text = str(full_prompt or "")
    history = []
    query = text.strip()
    if "User:" not in text:
        return query, history
    chunks = text.split("User:")
    for chunk in chunks[1:]:
        if "Assistant:" in chunk:
            user_part, assistant_part = chunk.split("Assistant:", 1)
            user_part = user_part.strip()
            assistant_part = assistant_part.strip()
            if user_part and assistant_part:
                history.append((user_part, assistant_part))
            elif user_part:
                query = user_part
        else:
            query = chunk.replace("SoulChat:", "").strip()
    return query, history[-6:]

@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_NAME, "service": "soulchat"}

@app.post("/v1/generate")
async def generate(request: Request):
    try:
        data = await request.json()
        prompt = str(data.get("prompt", ""))
        query, history = parse_prompt(prompt)
        if not query:
            return JSONResponse({"text": ""})

        max_length = int(data.get("max_length", 2048))
        top_p = float(data.get("top_p", 0.9))
        temperature = float(data.get("temperature", 0.7))
        response, _ = model.chat(
            tokenizer,
            query,
            history=history,
            max_length=max_length,
            top_p=top_p,
            temperature=temperature,
        )
        return JSONResponse({"text": str(response).strip(), "model": MODEL_NAME})
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# 5. Start ngrok + uvicorn
import uvicorn

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "").strip()
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("WARNING: NGROK_AUTH_TOKEN is empty. Add it as Kaggle Secret or os.environ before running.")

ngrok.kill()
public_url = ngrok.connect(8000).public_url
print("=" * 80)
print(f"SOULCHAT_NGROK_URL={public_url}")
print("Set this URL in backend .env, then benchmark will call only this SoulChat server.")
print("=" * 80)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio", log_level="info")
server = uvicorn.Server(config)
await server.serve()
